# 02 · Collect today's market data from public APIs

The Kaggle file stops in 2008. To see what changed, I pull current data straight from five public sources. Each pull is cached in `data/raw/` so the notebooks re-run offline.

| Source | What I take | Why |
|---|---|---|
| **NLM RxClass API** | Every diabetes ingredient and its ATC drug class | Groups drug names into classes from a drug database instead of my own labels |
| **CMS Part D Spending by Drug** (2020–2024) | Medicare spending, claims and patients per brand | Market map: where the money and prescriptions go today |
| **CMS Part D Prescribers by Provider & Drug** (2024) | Every prescriber's claims per diabetes drug, with specialty and state | Which doctors and specialties use the newer drugs (stands in for Symphony / APLD prescriber data) |
| **CMS Part D Prescribers by Geography & Drug** (2024) | Claims per state per drug | State-level adoption |
| **CDC PLACES** (2025 release) | Adult diabetes and obesity rates for every county | Where the disease burden is |
| **openFDA Drugs@FDA** | First FDA approval date per ingredient | Timeline of when each class arrived |

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # make src/ importable when run from notebooks/

import pandas as pd
from src import apis
from src.config import CLASS_ORDER, ADVANCED, PROCESSED, update_results
pd.set_option("display.width", 160)

## 1. Drug classes (NLM RxClass, ATC system)

I request the members of each diabetes ATC class. Three ingredients sit in the catch-all "other" class (A10BX), so I place them by mechanism: **tirzepatide** with GLP-1 drugs (it's a dual GIP/GLP-1 agonist), **pramlintide** as an amylin analog, and **repaglinide / nateglinide** as meglitinides.

In [2]:
ATC = ["A10A", "A10BA", "A10BB", "A10BF", "A10BG", "A10BH", "A10BJ", "A10BK", "A10BX"]
rx = apis.rxclass_members(ATC)

ATC_TO_CLASS = {"A10A": "Insulin", "A10BA": "Biguanide (metformin)", "A10BB": "Sulfonylurea",
                "A10BF": "Alpha-glucosidase inhibitor", "A10BG": "Thiazolidinedione", "A10BH": "DPP-4 inhibitor",
                "A10BJ": "GLP-1 / GIP-GLP-1 agonist", "A10BK": "SGLT2 inhibitor"}
OVERRIDE = {"tirzepatide": "GLP-1 / GIP-GLP-1 agonist", "pramlintide": "Amylin analog",
            "repaglinide": "Meglitinide", "nateglinide": "Meglitinide"}
rx["drug_class"] = rx.ingredient.map(OVERRIDE).fillna(rx.atc_id.map(ATC_TO_CLASS))
rx = rx.dropna(subset=["drug_class"])
# Insulin names come back as "insulin aspart, human"; keep the first two words for matching
rx["key"] = rx.ingredient.str.replace(",", "").str.split().str[:2].str.join(" ")
rx.loc[~rx.ingredient.str.startswith("insulin"), "key"] = rx.ingredient.str.split().str[0]
ingredient_class = dict(zip(rx.key, rx.drug_class))
rx.groupby("drug_class").ingredient.apply(lambda s: ", ".join(sorted(s))).reindex(CLASS_ORDER).dropna().to_frame()

,ingredient
drug_class,
Biguanide (metformin),"metformin, phenformin"
Sulfonylurea,"acetohexamide, carbutamide, chlorpropamide, gl..."
Thiazolidinedione,"pioglitazone, rosiglitazone, troglitazone"
Meglitinide,"nateglinide, repaglinide"
Alpha-glucosidase inhibitor,"acarbose, miglitol"
Insulin,"insulin aspart, human, insulin degludec, insul..."
DPP-4 inhibitor,"alogliptin, linagliptin, saxagliptin, sitaglip..."
SGLT2 inhibitor,"bexagliflozin, canagliflozin, dapagliflozin, e..."
GLP-1 / GIP-GLP-1 agonist,"albiglutide, dulaglutide, exenatide, liragluti..."


**Combination pills** (e.g. empagliflozin/metformin) are counted under their **newest** component, since that's the drug the combination exists to deliver.

In [3]:
PRIORITY = ["GLP-1 / GIP-GLP-1 agonist", "SGLT2 inhibitor", "DPP-4 inhibitor", "Insulin", "Amylin analog",
            "Thiazolidinedione", "Sulfonylurea", "Meglitinide", "Alpha-glucosidase inhibitor", "Biguanide (metformin)"]

def classify(name: str):
    """Map a drug name (CMS generic name or Kaggle column) to one drug class, or None."""
    n = name.lower()
    if n.startswith("insulin") and not any(w in n for w in ["pump", "device", "supplies", "pen,reusable", "pmp"]):
        found = {"Insulin"}
    else:
        parts = n.replace("-", "/").split("/")
        found = {ingredient_class.get(p.strip().replace(",", " ").split()[0]) for p in parts if p.strip()}
    if "lixisenatide" in n or "liraglutide" in n:
        found.add("GLP-1 / GIP-GLP-1 agonist")
    found.discard(None)
    return next((c for c in PRIORITY if c in found), None)

for t in ["Metformin HCl", "Empagliflozin/Metformin HCl", "Insulin Glargine/Lixisenatide", "glyburide-metformin",
          "Tirzepatide", "Syringe And Needle,Insulin,1ml", "Insulin Pump Cart,Cont Inf,Bt"]:
    print(f"{t:35s} -> {classify(t)}")

Metformin HCl                       -> Biguanide (metformin)
Empagliflozin/Metformin HCl         -> SGLT2 inhibitor
Insulin Glargine/Lixisenatide       -> GLP-1 / GIP-GLP-1 agonist
glyburide-metformin                 -> Sulfonylurea
Tirzepatide                         -> GLP-1 / GIP-GLP-1 agonist
Syringe And Needle,Insulin,1ml      -> None
Insulin Pump Cart,Cont Inf,Bt       -> None


## 2. Medicare Part D spending by drug, 2020–2024

In [4]:
spend = apis.cms_spending_by_drug()
spend["drug_class"] = spend.Gnrc_Name.map(classify)
diab = spend.dropna(subset=["drug_class"]).copy()
num = [c for c in diab.columns if c.startswith(("Tot_Spndng_", "Tot_Clms_", "Tot_Benes_"))]
diab[num] = diab[num].apply(pd.to_numeric, errors="coerce")
diab["weight_loss_brand"] = diab.Brnd_Name.isin(["Wegovy", "Zepbound", "Saxenda"])
print(len(spend), "brands in Part D;", len(diab), "are diabetes drugs")
diab.sort_values("Tot_Spndng_2024", ascending=False)[["Brnd_Name", "Gnrc_Name", "drug_class", "Tot_Spndng_2024", "Tot_Clms_2024"]].head(12)

3625 brands in Part D; 142 are diabetes drugs


,Brnd_Name,Gnrc_Name,drug_class,Tot_Spndng_2024,Tot_Clms_2024
2346,Ozempic,Semaglutide,GLP-1 / GIP-GLP-1 agonist,1.297030e+10,10417182
1601,Jardiance,Empagliflozin,SGLT2 inhibitor,1.143600e+10,11368280
2038,Mounjaro,Tirzepatide,GLP-1 / GIP-GLP-1 agonist,6.336943e+09,5105397
3219,Trulicity,Dulaglutide,GLP-1 / GIP-GLP-1 agonist,5.454135e+09,4305141
1163,Farxiga,Dapagliflozin Propanediol,SGLT2 inhibitor,5.290549e+09,5634650
1600,Januvia,Sitagliptin Phosphate,DPP-4 inhibitor,3.807855e+09,4027317
2744,Rybelsus,Semaglutide,GLP-1 / GIP-GLP-1 agonist,1.890082e+09,1251733
3134,Tradjenta,Linagliptin,DPP-4 inhibitor,1.127598e+09,1446832
3156,Tresiba Flextouch U-200,Insulin Degludec,Insulin,9.738841e+08,893822
3155,Tresiba Flextouch U-100,Insulin Degludec,Insulin,8.257979e+08,1281963


Wegovy, Zepbound and Saxenda share ingredients with diabetes drugs but are approved for weight loss (Medicare covers Wegovy for heart-risk reduction). I flag them and leave them out of the diabetes market so the numbers aren't inflated.

## 3. Every prescriber's diabetes claims (2024)

In [5]:
generics = sorted(diab.Gnrc_Name.unique())
generics = [g for g in generics if classify(g)]
print(len(generics), "generic names to pull")
rx_claims = apis.cms_prescriber_drug(generics)       # ~2.2M rows the first time, then read from cache
rx_claims["drug_class"] = rx_claims.Gnrc_Name.map(classify)
rx_claims = rx_claims[~rx_claims.Brnd_Name.isin(["Wegovy", "Zepbound", "Saxenda"])]
print(f"{len(rx_claims):,} prescriber-drug rows, {rx_claims.Prscrbr_NPI.nunique():,} prescribers")
rx_claims.drop(columns="Prscrbr_NPI").head()   # preview without prescriber IDs

62 generic names to pull


2,231,207 prescriber-drug rows, 316,488 prescribers


,Prscrbr_Type,Prscrbr_State_Abrvtn,Brnd_Name,Gnrc_Name,Tot_Clms,Tot_Benes,Tot_Drug_Cst,drug_class
0,Family Practice,FL,Acarbose,Acarbose,13,NaN,422.81,Alpha-glucosidase inhibitor
1,Internal Medicine,CA,Acarbose,Acarbose,12,NaN,800.78,Alpha-glucosidase inhibitor
2,Endocrinology,AL,Acarbose,Acarbose,26,NaN,1629.41,Alpha-glucosidase inhibitor
3,Internal Medicine,TX,Acarbose,Acarbose,26,11.0,1064.61,Alpha-glucosidase inhibitor
4,Family Practice,LA,Acarbose,Acarbose,14,NaN,280.57,Alpha-glucosidase inhibitor


CMS hides any row with fewer than 11 claims to protect patient privacy, so low-volume prescribers are missing for some drugs. That undercounts new-drug adoption among doctors who've only just started prescribing, which I'll note as a limitation.

## 4. Claims by state (2024)

In [6]:
state = apis.cms_state_drug(generics)
for c in ["Tot_Prscrbrs", "Tot_Clms", "Tot_Benes", "Tot_Drug_Cst"]:
    state[c] = pd.to_numeric(state[c], errors="coerce")
state["drug_class"] = state.Gnrc_Name.map(classify)
state = state[~state.Brnd_Name.isin(["Wegovy", "Zepbound", "Saxenda"])]
state.groupby("Prscrbr_Geo_Desc").Tot_Clms.sum().sort_values(ascending=False).head()

Prscrbr_Geo_Desc
California      12210639
Texas            8911759
Florida          8673705
New York         8477022
Pennsylvania     5737490
Name: Tot_Clms, dtype: int64

## 5. Disease burden by county (CDC PLACES)

In [7]:
places = apis.cdc_places_county()
wide = places.pivot_table(index=["stateabbr", "statedesc", "locationname", "locationid", "totalpop18plus"],
                          columns="measureid", values="data_value").reset_index()
wide["adults_with_diabetes"] = wide.totalpop18plus * wide.DIABETES / 100
print(len(wide), "counties; data year", places.year.unique())
wide.sort_values("DIABETES", ascending=False).head()

2956 counties; data year ['2023']


measureid,stateabbr,statedesc,locationname,locationid,totalpop18plus,DIABETES,OBESITY,adults_with_diabetes
61,AL,Alabama,Greene,01063,5755,27.1,52.7,1559.605
1469,MS,Mississippi,Sharkey,28125,2528,25.2,50.3,637.056
1052,LA,Louisiana,Tensas,22107,2991,25.1,48.4,750.741
415,GA,Georgia,Randolph,13243,4809,24.9,45.6,1197.441
1455,MS,Mississippi,Humphreys,28053,5419,24.7,52.4,1338.493


## 6. When each class arrived (openFDA)

In [8]:
ingredients = ["metformin", "glipizide", "glyburide", "glimepiride", "pioglitazone", "rosiglitazone", "acarbose",
               "repaglinide", "nateglinide", "sitagliptin", "saxagliptin", "linagliptin", "alogliptin",
               "exenatide", "liraglutide", "dulaglutide", "semaglutide", "tirzepatide",
               "canagliflozin", "dapagliflozin", "empagliflozin", "ertugliflozin", "insulin glargine", "insulin lispro"]
fda = apis.first_approvals(ingredients)
fda["drug_class"] = fda.ingredient.map(lambda i: "Insulin" if i.startswith("insulin") else ingredient_class.get(i))
first_by_class = fda.sort_values("first_approval").groupby("drug_class").first().reset_index()
first_by_class[["drug_class", "ingredient", "brand", "first_approval"]].sort_values("first_approval")

,drug_class,ingredient,brand,first_approval
5,Sulfonylurea,glyburide,GLYBURIDE,1984-05-01
3,Insulin,insulin lispro,"INSULIN LISPRO, INSULIN LISPRO JUNIOR KWIKPEN",1996-06-14
6,Thiazolidinedione,pioglitazone,ACTOS,1999-07-15
2,GLP-1 / GIP-GLP-1 agonist,exenatide,BYETTA,2005-04-28
0,Biguanide (metformin),metformin,"METFORMIN HYDROCHLORIDE, GLUMETZA",2005-06-03
1,DPP-4 inhibitor,sitagliptin,JANUVIA,2006-10-16
4,SGLT2 inhibitor,canagliflozin,INVOKANA,2013-03-29


openFDA only covers applications in its current database, so the oldest drugs (insulin, early sulfonylureas) can show a later date than their true first approval. I use these dates only to show the order in which the **newer** classes arrived, not as exact history.

## 7. What each API returns, and a privacy check

For the presentation: the exact request behind each source, a few real rows, and checks that nothing identifies a patient.

The prescriber API also offers each prescriber's **name and city**. I didn't request those columns (see `apis.cms_prescriber_drug`), and I never show the provider ID (NPI) on its own. CMS publishes this file for transparency about **prescribers**, and it contains no patient-level rows: every row is a total over a prescriber's patients.

In [9]:
import requests
offered = list(requests.get(apis.CMS.format(id=apis.CMS_PROVIDER_DRUG_2024), params={"size": 1}, timeout=60).json()[0])
pulled = [c for c in rx_claims.columns if c != "drug_class"]
not_pulled = [c for c in offered if c not in pulled]
print("Columns the API offers:", len(offered))
print("Columns I pulled:      ", pulled)
print("Left out (incl. names, city):", not_pulled)

Columns the API offers: 22
Columns I pulled:       ['Prscrbr_NPI', 'Prscrbr_Type', 'Prscrbr_State_Abrvtn', 'Brnd_Name', 'Gnrc_Name', 'Tot_Clms', 'Tot_Benes', 'Tot_Drug_Cst']
Left out (incl. names, city): ['Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City', 'Prscrbr_State_FIPS', 'Prscrbr_Type_Src', 'Tot_30day_Fills', 'Tot_Day_Suply', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes']


In [10]:
privacy_api = {
    "prescriber_columns_offered": offered,
    "prescriber_columns_pulled": pulled,
    "prescriber_min_claims_per_row": int(rx_claims.Tot_Clms.min()),          # CMS drops rows under 11 claims
    "prescriber_benes_suppressed_pct": round(100 * rx_claims.Tot_Benes.isna().mean(), 1),  # patient counts 1-10 blanked
    "state_min_prescribers_per_row": int(state.Tot_Prscrbrs.min()),
    "county_min_adults": int(wide.totalpop18plus.min()),
    "patient_level_rows": 0,
}
privacy_api

{'prescriber_columns_offered': ['Prscrbr_NPI',
  'Prscrbr_Last_Org_Name',
  'Prscrbr_First_Name',
  'Prscrbr_City',
  'Prscrbr_State_Abrvtn',
  'Prscrbr_State_FIPS',
  'Prscrbr_Type',
  'Prscrbr_Type_Src',
  'Brnd_Name',
  'Gnrc_Name',
  'Tot_Clms',
  'Tot_30day_Fills',
  'Tot_Day_Suply',
  'Tot_Drug_Cst',
  'Tot_Benes',
  'GE65_Sprsn_Flag',
  'GE65_Tot_Clms',
  'GE65_Tot_30day_Fills',
  'GE65_Tot_Drug_Cst',
  'GE65_Tot_Day_Suply',
  'GE65_Bene_Sprsn_Flag',
  'GE65_Tot_Benes'],
 'prescriber_columns_pulled': ['Prscrbr_NPI',
  'Prscrbr_Type',
  'Prscrbr_State_Abrvtn',
  'Brnd_Name',
  'Gnrc_Name',
  'Tot_Clms',
  'Tot_Benes',
  'Tot_Drug_Cst'],
 'prescriber_min_claims_per_row': 11,
 'prescriber_benes_suppressed_pct': np.float64(64.4),
 'state_min_prescribers_per_row': 1,
 'county_min_adults': 75,
 'patient_level_rows': 0}

- Every prescriber row has **at least 11 claims** (CMS suppresses smaller ones), and CMS blanks the patient count whenever it's 1–10, so small groups of patients can't be singled out.
- State rows are totals across every prescriber in a state, and county rows are survey-based **percentages** for a whole county, not records of people.
- RxClass and openFDA describe drugs, not people.

In [11]:
def sample(df, cols, n=3, sort=None, ascending=False):
    d = df.sort_values(sort, ascending=ascending) if sort else df
    return d[cols].head(n).astype(str).to_dict("records")

ozempic = spend[spend.Brnd_Name == "Ozempic"].iloc[0]
api_samples = [
    {"name": "CMS Part D Prescribers by Provider & Drug (2024)", "host": "data.cms.gov",
     "request": f"GET /data-api/v1/dataset/{apis.CMS_PROVIDER_DRUG_2024[:8]}…/data?filter[Gnrc_Name]=Semaglutide&column=Prscrbr_Type,Prscrbr_State_Abrvtn,Brnd_Name,…&size=5000&offset=0",
     "loop": f"Paged 5,000 rows at a time for each of {len(generics)} diabetes generic names, 8 requests in parallel",
     "one_row": "One prescriber × one drug: that prescriber's 2024 Medicare claims for it",
     "rows": sample(rx_claims[rx_claims.Gnrc_Name == "Semaglutide"].sample(3, random_state=3),   # random rows, not the top prescriber
                   ["Prscrbr_Type", "Prscrbr_State_Abrvtn", "Brnd_Name", "Gnrc_Name", "Tot_Clms", "Tot_Benes", "Tot_Drug_Cst"]),
     "total": int(len(rx_claims))},
    {"name": "CMS Part D Spending by Drug (2020–2024)", "host": "data.cms.gov",
     "request": f"GET /data-api/v1/dataset/{apis.CMS_SPENDING_2024[:8]}…/data?filter[Mftr_Name]=Overall&size=5000",
     "loop": "One paged pull of all 3,625 brands, then kept the 142 diabetes brands",
     "one_row": "One brand: national Medicare spending, claims and patients for each year 2020–2024",
     "rows": sample(diab, ["Brnd_Name", "Gnrc_Name", "drug_class", "Tot_Spndng_2024", "Tot_Clms_2024", "Tot_Benes_2024"], sort="Tot_Spndng_2024"),
     "raw_json": {k: str(ozempic[k]) for k in ["Brnd_Name", "Gnrc_Name", "Mftr_Name", "Tot_Spndng_2023", "Tot_Spndng_2024", "Tot_Clms_2024", "Tot_Benes_2024", "Avg_Spnd_Per_Clm_2024"]},
     "total": int(len(diab))},
    {"name": "CMS Part D Prescribers by Geography & Drug (2024)", "host": "data.cms.gov",
     "request": f"GET /data-api/v1/dataset/{apis.CMS_GEO_DRUG_2024[:8]}…/data?filter[Prscrbr_Geo_Lvl]=State&filter[Gnrc_Name]=Empagliflozin",
     "loop": f"One request per diabetes generic name ({len(generics)})",
     "one_row": "One state × one brand: total prescribers, claims and cost",
     "rows": sample(state[state.Gnrc_Name == "Empagliflozin"], ["Prscrbr_Geo_Desc", "Brnd_Name", "Tot_Prscrbrs", "Tot_Clms", "Tot_Drug_Cst"], sort="Tot_Clms"),
     "total": int(len(state))},
    {"name": "CDC PLACES county data (2025 release)", "host": "data.cdc.gov",
     "request": "GET /resource/swc5-untb.json?$where=datavaluetypeid='CrdPrv' AND measureid in('DIABETES','OBESITY')&$limit=50000",
     "loop": "One request; Socrata query language does the filtering server-side",
     "one_row": "One county: % of adults with diagnosed diabetes and obesity (survey-based model estimates)",
     "rows": sample(wide.rename(columns={"DIABETES": "diabetes_pct", "OBESITY": "obesity_pct"}),
                    ["statedesc", "locationname", "totalpop18plus", "diabetes_pct", "obesity_pct"], sort="totalpop18plus"),
     "total": int(len(wide))},
    {"name": "NLM RxClass (ATC classes)", "host": "rxnav.nlm.nih.gov",
     "request": "GET /REST/rxclass/classMembers.json?classId=A10BK&relaSource=ATC&ttys=IN",
     "loop": f"One request per ATC class ({len(ATC)})",
     "one_row": "One ingredient and the drug class it belongs to",
     "rows": sample(rx[rx.drug_class.isin(["SGLT2 inhibitor", "GLP-1 / GIP-GLP-1 agonist"])], ["atc_id", "atc_name", "ingredient", "drug_class"], n=4),
     "total": int(len(rx))},
    {"name": "openFDA Drugs@FDA", "host": "api.fda.gov",
     "request": 'GET /drug/drugsfda.json?search=openfda.generic_name:"empagliflozin"&limit=100',
     "loop": f"One request per ingredient ({len(ingredients)}), keeping the earliest original NDA/BLA approval",
     "one_row": "One ingredient: first approval date and application number",
     "rows": sample(fda.assign(first_approval=fda.first_approval.dt.date), ["ingredient", "brand", "application", "first_approval"], n=4, sort="first_approval"),
     "total": int(len(fda))},
]
pd.DataFrame(api_samples)[["name", "request", "total"]]

,name,request,total
0,CMS Part D Prescribers by Provider & Drug (2024),GET /data-api/v1/dataset/9552739e…/data?filter...,2231207
1,CMS Part D Spending by Drug (2020–2024),GET /data-api/v1/dataset/7e0b4365…/data?filter...,142
2,CMS Part D Prescribers by Geography & Drug (2024),GET /data-api/v1/dataset/c8ea3f8e…/data?filter...,6615
3,CDC PLACES county data (2025 release),GET /resource/swc5-untb.json?$where=datavaluet...,2956
4,NLM RxClass (ATC classes),GET /REST/rxclass/classMembers.json?classId=A1...,48
5,openFDA Drugs@FDA,GET /drug/drugsfda.json?search=openfda.generic...,19


In [12]:
rx_claims.to_parquet(PROCESSED / "prescriber_claims.parquet", index=False)
state.to_parquet(PROCESSED / "state_claims.parquet", index=False)
diab.to_parquet(PROCESSED / "spending_by_brand.parquet", index=False)
wide.to_parquet(PROCESSED / "county_burden.parquet", index=False)
fda.to_parquet(PROCESSED / "fda_approvals.parquet", index=False)
pd.DataFrame({"ingredient_key": list(ingredient_class), "drug_class": list(ingredient_class.values())}).to_parquet(
    PROCESSED / "ingredient_class.parquet", index=False)

update_results("sources", [
    {"name": "Kaggle · Diabetes 130-US Hospitals (1999–2008)", "rows": 101766, "kind": "Downloaded dataset"},
    {"name": "CMS Part D Prescribers by Provider & Drug (2024)", "rows": int(len(rx_claims)), "kind": "API"},
    {"name": "CMS Part D Spending by Drug (2020–2024)", "rows": int(len(diab)), "kind": "API"},
    {"name": "CMS Part D Prescribers by Geography & Drug (2024)", "rows": int(len(state)), "kind": "API"},
    {"name": "CDC PLACES county data (2025 release)", "rows": int(len(wide)), "kind": "API"},
    {"name": "NLM RxClass (ATC classes)", "rows": int(len(rx)), "kind": "API"},
    {"name": "openFDA Drugs@FDA (approval dates)", "rows": int(len(fda)), "kind": "API"},
])
update_results("api_samples", api_samples)
update_results("privacy_api", privacy_api)
update_results("approvals", [
    {"drug_class": r.drug_class, "ingredient": r.ingredient, "brand": r.brand, "year": int(r.first_approval.year)}
    for r in first_by_class.itertuples()])
print("saved")

saved
